In [11]:
import os
import glob
import pandas as pd
import numpy as np

# 1. LOAD DỮ LIỆU THÀNH LONG-FORMAT DATAFRAME
# Theo tài liệu của Chronos-2 (predict_df), model yêu cầu truyền vào 1 dataframe duy nhất
# gồm 3 cột bắt buộc: id_column, timestamp_column, và target
# Model KHÔNG nhận Dataframe Wide-format (Mỗi cột 1 mã) 
# vì nó cần sử dụng Cross-learning (batch size).

data_dir = '../../data/raw/'
csv_files = glob.glob(os.path.join(data_dir, '*.csv'))

dfs = []
for file in csv_files:
    ticker = os.path.basename(file).split('.')[0]
    df = pd.read_csv(file, parse_dates=['Date'])
    df['id'] = ticker # Cột ID cho Chronos-2
    dfs.append(df)

# Gộp nối bằng pd.concat(axis=0) thành định dạng Dọc (Long-format)
df_all = pd.concat(dfs, ignore_index=True)
df_all.rename(columns={'Date': 'timestamp'}, inplace=True)
df_all.sort_values(by=['id', 'timestamp'], inplace=True)
df_all.reset_index(drop=True, inplace=True)

print("Kích thước dữ liệu gộp Long-Format:", df_all.shape)
display(df_all.head())
df_all.to_csv('../../data/processed/merged_data.csv', index=False)

Kích thước dữ liệu gộp Long-Format: (25736, 9)


,timestamp,Open,High,Low,Close,Volume,id,USD_VND,CPI
0,2013-01-02,3.963,4.109,3.963,4.087,380320.0,FPT,NaN,NaN
1,2013-01-03,4.087,4.109,3.963,4.031,366960.0,FPT,NaN,NaN
2,2013-01-04,4.008,4.087,3.997,4.087,278020.0,FPT,NaN,NaN
3,2013-01-07,4.064,4.098,4.042,4.064,204570.0,FPT,NaN,NaN
4,2013-01-08,4.076,4.267,4.064,4.267,535860.0,FPT,NaN,NaN


In [12]:
# 2. FEATURE ENGINEERING: CÁC BIẾN THỜI GIAN VÀ ĐÁO HẠN PHÁI SINH
# Các biến này là Known Future Covariates (Biết chắc chắn trong tương lai)

def is_derivatives_expiration(date):
    """
    Ngày đáo hạn phái sinh VN30: Thứ Năm tuần thứ 3 trong tháng.
    Thứ 5 tương ứng với weekday() == 3.
    Tuần thứ 3 có ngày rơi vào khoảng từ 15 đến 21 của tháng.
    """
    if date.weekday() == 3 and (15 <= date.day <= 21):
        return 1
    return 0

# Thêm biến Đáo hạn phái sinh
df_all['is_derivative_exp'] = df_all['timestamp'].apply(is_derivatives_expiration)

# Thêm các biến lịch/chu kỳ (Thứ trong tuần, Tháng, Quý)
df_all['day_of_week'] = df_all['timestamp'].dt.dayofweek # 0=Thứ 2, 4=Thứ 6
df_all['quarter'] = df_all['timestamp'].dt.quarter
df_all['month'] = df_all['timestamp'].dt.month

print("Số ngày đáo hạn phái sinh phát hiện được trong mỗi mã:")
print(df_all.groupby('id')['is_derivative_exp'].sum())
display(df_all[df_all['is_derivative_exp'] == 1].head())

Số ngày đáo hạn phái sinh phát hiện được trong mỗi mã:
id
FPT         153
HPG         153
SSI         153
VCB         153
VHM          91
VN30         98
VNINDEX      98
VNM         153
macro_vn    156
Name: is_derivative_exp, dtype: int64


,timestamp,Open,High,Low,Close,Volume,id,USD_VND,CPI,is_derivative_exp,day_of_week,quarter,month
11,2013-01-17,4.323,4.345,4.278,4.278,292200.0,FPT,NaN,NaN,1,3,1,1
31,2013-02-21,4.435,4.446,4.334,4.334,307570.0,FPT,NaN,NaN,1,3,1,2
51,2013-03-21,4.199,4.334,4.188,4.289,274540.0,FPT,NaN,NaN,1,3,1,3
71,2013-04-18,4.222,4.244,4.143,4.154,135860.0,FPT,NaN,NaN,1,3,2,4
87,2013-05-16,4.379,4.379,4.323,4.356,129340.0,FPT,NaN,NaN,1,3,2,5


In [13]:
# 3. MACROSCOPIC FEATURES / CHỈ BÁO KỸ THUẬT (Past Covariates)
# (LƯU Ý: Đây là code mẫu, khi bạn có data vĩ mô, hãy merge bằng "left join" và "forward fill")

# a. Thêm Lợi suất (Returns) - Có thể coi đây là Target thay vì Giá Close
df_all['log_return'] = df_all.groupby('id')['Close'].transform(lambda x: np.log(x / x.shift(1)))

# b. Thêm Tương quan với thị trường chung (Biến VNINDEX)
# Giả sử bạn lấy VNINDEX làm tham chiếu.
vnindex_data = df_all[df_all['id'] == 'VNINDEX'][['timestamp', 'Close']]
vnindex_data.rename(columns={'Close': 'VNI_Close'}, inplace=True)

df_all = pd.merge(df_all, vnindex_data, on='timestamp', how='left')

# Tính ROI thị trường VNINDEX
df_all['VNI_Return'] = df_all.groupby('id')['VNI_Close'].transform(lambda x: x.pct_change())

# ==========================================
# c. XỬ LÝ VÀ HỢP NHẤT DỮ LIỆU MACRO_VN.CSV
# ==========================================
print("Đang xử lý dữ liệu vĩ mô macro_vn.csv...")
macro_filepath = '../../data/raw/macro_vn.csv'
if os.path.exists(macro_filepath):
    # Đọc dữ liệu vĩ mô
    macro_df = pd.read_csv(macro_filepath, parse_dates=['Date'])
    
    # 1. Đổi tên một số cột có khoảng trắng
    macro_df.rename(columns={'Interest Rate ': 'Interest_Rate'}, inplace=True)
    
    # 2. Điền khuyết (Forward Fill) trực tiếp cho các biến Tần suất Hàng ngày (Daily)
    # Tỷ giá USD_VND và Lãi suất nếu có đứt gãy (nghỉ lễ, cuối tuần) thì ffill là an toàn.
    daily_macro_cols = ['USD_VND', 'Interest_Rate']
    for col in daily_macro_cols:
        if col in macro_df.columns:
            macro_df[col] = macro_df[col].ffill()
            
    # 3. Xử lý an toàn cho biến Tần suất Tháng/Quý (CPI, GDP, FDI) - TRÁNH LOOK-AHEAD BIAS
    # Chỉ bốc những dòng có dữ liệu -> đẩy lùi (shift 1) để nó xuất hiện ở kỳ tiếp theo -> trải dài mảng trống bằng ffill
    period_macro_cols = ['CPI', 'GDP', 'FDI']
    for col in period_macro_cols:
        if col in macro_df.columns:
            # Dropna để lọc ra các ngày công bố định kỳ -> Shift 1 để trễ 1 kỳ -> Khôi phục index ngày -> Trải dài ffill
            safe_delayed_series = macro_df[col].dropna().shift(1).reindex(macro_df.index)
            macro_df[col] = safe_delayed_series.ffill()
            
    # 4. Hợp nhất `macro_df` vào dữ liệu chính `df_all` bằng ngàm 'timestamp' (left join)
    df_all = pd.merge(df_all, macro_df, left_on='timestamp', right_on='Date', how='left')
    df_all.drop(columns=['Date'], inplace=True) # Xóa cột Date thừa sau khi đính kèm
    
    # Xóa các dòng bị thiếu Vĩ mô hoặc backfill/ffill lần cuối để liền mạch
    existing_macro_cols = [c for c in daily_macro_cols + period_macro_cols if c in df_all.columns]
    df_all[existing_macro_cols] = df_all[existing_macro_cols].ffill().bfill()
    print(f"✅ Đã hợp nhất thành công biến vĩ mô: {existing_macro_cols}")
else:
    print(f"Cảnh báo: Không thể nạp được file {macro_filepath}")

# ==========================================

# Xử lý NaN chung (do hàm tỷ suất log_return có dòng đầu tiên là NaN, và quá trình shift)
df_all.dropna(subset=['log_return', 'VNI_Return'], inplace=True)

print("\nCơ sở dữ liệu hoàn thiện:")
display(df_all.tail())

Đang xử lý dữ liệu vĩ mô macro_vn.csv...
✅ Đã hợp nhất thành công biến vĩ mô: []

Cơ sở dữ liệu hoàn thiện:


C:\Users\admin\AppData\Local\Temp\ipykernel_12452\1944613567.py:15: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_all['VNI_Return'] = df_all.groupby('id')['VNI_Close'].transform(lambda x: x.pct_change())


,timestamp,Open,High,Low,Close,Volume,id,USD_VND_x,CPI_x,is_derivative_exp,day_of_week,quarter,month,log_return,VNI_Close,VNI_Return,USD_VND_y,CPI_y
22296,2025-12-25,61.6,62.3,61.3,61.3,2059600.0,VNM,NaN,NaN,0,3,4,12,-0.003257,1742.85,-0.022420,NaN,NaN
22297,2025-12-26,61.6,62.3,61.2,61.5,2399200.0,VNM,NaN,NaN,0,4,4,12,0.003257,1729.80,-0.007488,26277.0,104.08
22298,2025-12-29,61.5,62.2,61.4,62.1,1943000.0,VNM,NaN,NaN,0,0,4,12,0.009709,1754.84,0.014476,26255.0,104.08
22299,2025-12-30,62.1,62.3,61.7,61.8,1581000.0,VNM,NaN,NaN,0,1,4,12,-0.004843,1766.90,0.006872,26245.0,104.08
22300,2025-12-31,61.9,62.0,61.2,61.2,1879600.0,VNM,NaN,NaN,0,2,4,12,-0.009756,1784.49,0.009955,26245.0,104.08


In [14]:
df_all.to_csv('../../data/processed/merged_data_with_features.csv', index=False)

In [16]:
# Chỉ giữ lại các mã cổ phiếu thực sự, loại bỏ VNINDEX và VN30 khỏi cột id cần dự báo
df_all = df_all[~df_all['id'].isin(['VNINDEX', 'VN30'])]
df_all.to_csv('../../data/processed/merged_data_with_features.csv', index=False)

In [17]:
# 5. EDA & BỔ SUNG CHỈ BÁO KỸ THUẬT (TECHNICAL INDICATORS: MA, RSI)
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

file_path = '../../data/processed/merged_data_with_features.csv'
print(f"Đang đọc dữ liệu từ: {file_path}\n")
df_feat = pd.read_csv(file_path, parse_dates=['timestamp'])

print("--- 1. KIỂM TRA DỮ LIỆU THIẾU (MISSING VALUES) GỐC ---")
print(df_feat.isnull().sum()[df_feat.isnull().sum() > 0]) # Chỉ in các cột có NaN

# ==========================================
# THÊM CÁC CHỈ BÁO KỸ THUẬT (RSI, MA)
# ==========================================
print("\nĐang tính toán các chỉ báo Phân tích kỹ thuật (MA20, MA50, RSI_14)...")

# Tính Moving Average (MA)
df_feat['MA20'] = df_feat.groupby('id')['Close'].transform(lambda x: x.rolling(window=20).mean())
df_feat['MA50'] = df_feat.groupby('id')['Close'].transform(lambda x: x.rolling(window=50).mean())

# Hàm tính Relative Strength Index (RSI - 14 ngày)
def calculate_rsi(series, period=14):
    delta = series.diff()
    # Dùng EMA (Exponential Moving Average) để mượt hơn, nhưng ở đây dùng SMA cho chạy nhanh
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

df_feat['RSI_14'] = df_feat.groupby('id')['Close'].transform(calculate_rsi)

# ==========================================
# XỬ LÝ DỮ LIỆU THIẾU PHÁT SINH DOT TÍNH TOÁN
# ==========================================
# Việc dùng .rolling(20) sẽ khiến 19 dòng đầu tiên của mỗi mã cổ phiếu bị NaN.
# Cách tốt nhất để giữ số lượng dòng: fill lấp ngược (Back-fill) những ngày đầu tiên bằng giá trị của ngày đầu tiên có MA/RSI
print("\nĐang lấp đầy các dữ liệu thiếu do Rolling window...")
df_feat = df_feat.groupby('id', group_keys=False).apply(lambda x: x.bfill().ffill())

# Đảm bảo không còn bất kỳ ô trống nào
print("\n--- 2. KIỂM TRA DỮ LIỆU THIẾU SAU KHI XỬ LÝ ---")
missing_after = df_feat.isnull().sum().sum()
print(f"Tổng số ô trống hiện tại: {missing_after}")

# Lưu ra file cuối cùng để đưa vào Chronos
final_path = '../../data/processed/merged_data_final.csv'
df_feat.to_csv(final_path, index=False)
print(f"\n✅ Đã lưu tập dữ liệu chuẩn cuối cùng tại: {final_path}")

# Xem thử kết quả
display(df_feat[['id', 'timestamp', 'Close', 'MA20', 'MA50', 'RSI_14']].tail(10))

Đang đọc dữ liệu từ: ../../data/processed/merged_data_with_features.csv

--- 1. KIỂM TRA DỮ LIỆU THIẾU (MISSING VALUES) GỐC ---
USD_VND_y    12
CPI_y        12
dtype: int64

Đang tính toán các chỉ báo Phân tích kỹ thuật (MA20, MA50, RSI_14)...

Đang lấp đầy các dữ liệu thiếu do Rolling window...

--- 2. KIỂM TRA DỮ LIỆU THIẾU SAU KHI XỬ LÝ ---
Tổng số ô trống hiện tại: 0

✅ Đã lưu tập dữ liệu chuẩn cuối cùng tại: ../../data/processed/merged_data_final.csv


,id,timestamp,Close,MA20,MA50,RSI_14
12334,VNM,2025-12-18,63.9,62.935,60.45686,49.629630
12335,VNM,2025-12-19,64.0,63.140,60.54478,46.456693
12336,VNM,2025-12-22,63.2,63.150,60.58612,43.283582
12337,VNM,2025-12-23,62.6,63.185,60.62502,42.647059
12338,VNM,2025-12-24,61.5,63.120,60.64000,42.962963
12339,VNM,2025-12-25,61.3,63.085,60.64200,42.335766
12340,VNM,2025-12-26,61.5,62.960,60.69600,42.753623
12341,VNM,2025-12-29,62.1,62.820,60.83800,48.507463
12342,VNM,2025-12-30,61.8,62.660,60.94400,45.038168
12343,VNM,2025-12-31,61.2,62.490,61.04200,48.760331
